In [27]:
pip install pymysql requests python-dotenv pymongo beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\yoona\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [ ]:
import os
import time
from datetime import datetime

import pymysql
import requests
from dotenv import load_dotenv


load_dotenv()


# =========================================================
# 기본 설정
# =========================================================

BASE_URL = "http://192.168.0.51:4000"

PUBLIC_KEY_URL = f"{BASE_URL}/api/v1/public-key"
CARS_URL = f"{BASE_URL}/api/v1/cars"

# 최신 10,000건
PAGE_SIZE = 100
MAX_PAGES = 100


MYSQL_CONFIG = {
    "host": os.getenv("MYSQL_HOST"),
    "port": int(os.getenv("MYSQL_PORT", 3306)),
    "user": os.getenv("MYSQL_USER"),
    "password": os.getenv("MYSQL_PASSWORD"),
    "database": os.getenv("MYSQL_DATABASE"),
    "charset": "utf8mb4",
    "cursorclass": pymysql.cursors.DictCursor,
    "autocommit": False,
}


# =========================================================
# 1. API KEY 조회
# =========================================================

def get_api_key():

    response = requests.get(
        PUBLIC_KEY_URL,
        timeout=10
    )

    response.raise_for_status()

    body = response.json()

    api_key = body["data"]["current"]["api_key"]

    print("[API KEY] 현재 API Key 조회 완료")

    return api_key


# =========================================================
# 2. API 요청
#    403 / 429 / Timeout 대응
# =========================================================

def request_api(url, api_key):

    max_retries = 10
    retry_count = 0

    while True:

        try:

            headers = {
                "X-API-Key": api_key
            }

            response = requests.get(
                url,
                headers=headers,
                timeout=(10, 30)
            )

            # -----------------------------------------
            # API Key 변경
            # -----------------------------------------

            if response.status_code == 403:

                retry_count += 1

                if retry_count > max_retries:
                    response.raise_for_status()

                print(
                    "[WARN] API Key 변경 감지 "
                    "-> 새 Key 조회"
                )

                api_key = get_api_key()

                time.sleep(2)

                continue

            # -----------------------------------------
            # Rate Limit
            # -----------------------------------------

            if response.status_code == 429:

                retry_count += 1

                if retry_count > max_retries:
                    response.raise_for_status()

                retry_after = response.headers.get(
                    "Retry-After"
                )

                if retry_after:

                    try:
                        wait_seconds = int(retry_after)

                    except ValueError:
                        wait_seconds = 10

                else:

                    wait_seconds = min(
                        5 * retry_count,
                        60
                    )

                print(
                    f"[429] 요청 제한 "
                    f"-> {wait_seconds}초 대기 "
                    f"({retry_count}/{max_retries})"
                )

                time.sleep(wait_seconds)

                continue

            # -----------------------------------------
            # 정상 또는 기타 HTTP 오류
            # -----------------------------------------

            response.raise_for_status()

            return response.json(), api_key

        # ---------------------------------------------
        # Timeout / Network Error
        # ---------------------------------------------

        except (
            requests.exceptions.ConnectTimeout,
            requests.exceptions.ReadTimeout,
            requests.exceptions.ConnectionError
        ) as e:

            retry_count += 1

            if retry_count > max_retries:
                raise

            wait_seconds = min(
                5 * retry_count,
                60
            )

            print(
                f"[NETWORK] {type(e).__name__} "
                f"-> {wait_seconds}초 후 재시도 "
                f"({retry_count}/{max_retries})"
            )

            time.sleep(wait_seconds)


# =========================================================
# 3. 유틸
# =========================================================

def value_from(obj, *keys, default=None):

    if not isinstance(obj, dict):
        return default

    for key in keys:

        value = obj.get(key)

        if value is not None:
            return value

    return default


def normalize_date(value):

    if not value:
        return None

    if isinstance(value, str):
        return value[:10]

    return value


# =========================================================
# 4. 차량 JSON 정규화
# =========================================================

def normalize_car(raw):

    brand = raw.get("brand") or {}
    model = raw.get("model") or {}
    dealer = raw.get("dealer") or {}
    area = raw.get("businessArea") or {}
    location = raw.get("location") or {}

    return {

        "car_id": raw.get("id"),

        "listing_number": raw.get(
            "listingNumber"
        ),

        # 실제 JSON 구조 확인 완료
        "dealer_id": dealer.get("code"),

        # 실제 JSON 구조 확인 완료
        "business_area_code": area.get("id"),

        "brand": value_from(
            brand,
            "name",
            default=brand if isinstance(brand, str) else None
        ),

        "model": value_from(
            model,
            "name",
            default=model if isinstance(model, str) else None
        ),

        "trim": raw.get("trim"),

        "model_year": raw.get("modelYear"),

        "first_registration_date": normalize_date(
            raw.get("firstRegistrationDate")
            or raw.get("firstRegisteredAt")
        ),

        "mileage_km": raw.get("mileageKm"),

        "price": raw.get("price"),

        "currency": raw.get("currency"),

        "fuel_type": (
            raw.get("fuelType")
            or raw.get("fuel")
        ),

        "transmission": raw.get(
            "transmission"
        ),

        "color": raw.get("color"),

        "displacement_cc": (
            raw.get("displacementCc")
            or raw.get("engineDisplacementCc")
        ),

        "status": raw.get("status"),

        "accident_count": raw.get(
            "accidentCount"
        ),

        "owner_change_count": raw.get(
            "ownerChangeCount"
        ),

        "inspection_status": raw.get(
            "inspectionStatus"
        ),

        "province": value_from(
            location,
            "province",
            "sido",
            "region"
        ),

        "city": value_from(
            location,
            "city",
            "sigungu",
            "district"
        ),

        "listing_date": normalize_date(
            raw.get("listingDate")
            or raw.get("registeredDate")
        )
    }


# =========================================================
# 5. 테이블 생성
# =========================================================

def create_tables(conn):

    with conn.cursor() as cursor:

        # -----------------------------------------
        # business_areas
        # -----------------------------------------

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS business_areas (

            business_area_code VARCHAR(100)
                PRIMARY KEY,

            business_area_name VARCHAR(255),

            dealer_id VARCHAR(100),

            dealer_name VARCHAR(100),

            department VARCHAR(255),

            position VARCHAR(100)

        ) ENGINE=InnoDB
        DEFAULT CHARSET=utf8mb4;
        """)

        # -----------------------------------------
        # cars
        # -----------------------------------------

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS cars (

            car_id BIGINT PRIMARY KEY,

            listing_number VARCHAR(100)
                NOT NULL UNIQUE,

            dealer_id VARCHAR(100),

            business_area_code VARCHAR(100),

            brand VARCHAR(100),

            model VARCHAR(150),

            trim VARCHAR(150),

            model_year INT,

            first_registration_date DATE,

            mileage_km INT,

            price BIGINT,

            currency VARCHAR(20),

            fuel_type VARCHAR(50),

            transmission VARCHAR(50),

            color VARCHAR(50),

            displacement_cc INT,

            status VARCHAR(50),

            accident_count INT,

            owner_change_count INT,

            inspection_status VARCHAR(100),

            province VARCHAR(100),

            city VARCHAR(100),

            listing_date DATE,

            CONSTRAINT fk_car_business_area
                FOREIGN KEY (business_area_code)
                REFERENCES business_areas(
                    business_area_code
                )

        ) ENGINE=InnoDB
        DEFAULT CHARSET=utf8mb4;
        """)

        # -----------------------------------------
        # crawl_logs
        # -----------------------------------------

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS crawl_logs (

            log_id BIGINT AUTO_INCREMENT
                PRIMARY KEY,

            source_type VARCHAR(20),

            source_name VARCHAR(255),

            started_at DATETIME,

            finished_at DATETIME,

            fetched_count INT DEFAULT 0,

            inserted_count INT DEFAULT 0,

            updated_count INT DEFAULT 0,

            failed_count INT DEFAULT 0,

            status VARCHAR(30),

            error_message TEXT

        ) ENGINE=InnoDB
        DEFAULT CHARSET=utf8mb4;
        """)

    conn.commit()


# =========================================================
# 6. business_areas UPSERT
# =========================================================

def upsert_business_area(cursor, raw):

    area = raw.get("businessArea") or {}
    dealer = raw.get("dealer") or {}

    business_area_code = area.get("id")
    business_area_name = area.get("name")

    dealer_id = dealer.get("code")

    # 마스킹 이름
    dealer_name = dealer.get("displayName")

    department = dealer.get("department")
    position = dealer.get("position")

    if not business_area_code:
        return

    sql = """
    INSERT INTO business_areas (

        business_area_code,
        business_area_name,
        dealer_id,
        dealer_name,
        department,
        position

    )
    VALUES (
        %s, %s, %s, %s, %s, %s
    )

    ON DUPLICATE KEY UPDATE

        business_area_name =
            VALUES(business_area_name),

        dealer_id =
            VALUES(dealer_id),

        dealer_name =
            VALUES(dealer_name),

        department =
            VALUES(department),

        position =
            VALUES(position)
    """

    values = (
        business_area_code,
        business_area_name,
        dealer_id,
        dealer_name,
        department,
        position
    )

    cursor.execute(
        sql,
        values
    )


# =========================================================
# 7. cars UPSERT
# =========================================================

def upsert_car(cursor, car):

    cursor.execute(
        """
        SELECT car_id
        FROM cars
        WHERE car_id = %s
        """,
        (car["car_id"],)
    )

    exists = cursor.fetchone()

    sql = """
    INSERT INTO cars (

        car_id,
        listing_number,
        dealer_id,
        business_area_code,

        brand,
        model,
        trim,
        model_year,
        first_registration_date,

        mileage_km,
        price,
        currency,
        fuel_type,
        transmission,
        color,
        displacement_cc,

        status,

        accident_count,
        owner_change_count,
        inspection_status,

        province,
        city,

        listing_date
    )

    VALUES (

        %s, %s, %s, %s,

        %s, %s, %s, %s, %s,

        %s, %s, %s, %s, %s, %s, %s,

        %s,

        %s, %s, %s,

        %s, %s,

        %s
    )

    ON DUPLICATE KEY UPDATE

        dealer_id =
            VALUES(dealer_id),

        business_area_code =
            VALUES(business_area_code),

        brand =
            VALUES(brand),

        model =
            VALUES(model),

        trim =
            VALUES(trim),

        model_year =
            VALUES(model_year),

        first_registration_date =
            VALUES(first_registration_date),

        mileage_km =
            VALUES(mileage_km),

        price =
            VALUES(price),

        currency =
            VALUES(currency),

        fuel_type =
            VALUES(fuel_type),

        transmission =
            VALUES(transmission),

        color =
            VALUES(color),

        displacement_cc =
            VALUES(displacement_cc),

        status =
            VALUES(status),

        accident_count =
            VALUES(accident_count),

        owner_change_count =
            VALUES(owner_change_count),

        inspection_status =
            VALUES(inspection_status),

        province =
            VALUES(province),

        city =
            VALUES(city),

        listing_date =
            VALUES(listing_date)
    """

    values = (
        car["car_id"],
        car["listing_number"],
        car["dealer_id"],
        car["business_area_code"],

        car["brand"],
        car["model"],
        car["trim"],
        car["model_year"],
        car["first_registration_date"],

        car["mileage_km"],
        car["price"],
        car["currency"],
        car["fuel_type"],
        car["transmission"],
        car["color"],
        car["displacement_cc"],

        car["status"],

        car["accident_count"],
        car["owner_change_count"],
        car["inspection_status"],

        car["province"],
        car["city"],

        car["listing_date"]
    )

    cursor.execute(
        sql,
        values
    )

    return (
        "updated"
        if exists
        else "inserted"
    )


# =========================================================
# 8. crawl_logs 저장
# =========================================================

def write_log(
    conn,
    started_at,
    finished_at,
    fetched,
    inserted,
    updated,
    failed,
    status,
    error_message=None
):

    sql = """
    INSERT INTO crawl_logs (

        source_type,
        source_name,
        started_at,
        finished_at,
        fetched_count,
        inserted_count,
        updated_count,
        failed_count,
        status,
        error_message

    )

    VALUES (
        %s, %s,
        %s, %s,
        %s, %s, %s, %s,
        %s, %s
    )
    """

    values = (
        "API",
        "AutoData Lab Cars Initial",

        started_at,
        finished_at,

        fetched,
        inserted,
        updated,
        failed,

        status,
        error_message
    )

    with conn.cursor() as cursor:

        cursor.execute(
            sql,
            values
        )

    conn.commit()


# =========================================================
# 9. 최신 10,000건 조회 + 즉시 적재
# =========================================================

def load_initial_cars(
    conn,
    cursor,
    api_key
):

    fetched = 0
    inserted = 0
    updated = 0
    failed = 0

    # 중복 여부 검증용
    seen_car_ids = set()

    for page in range(
        1,
        MAX_PAGES + 1
    ):

        url = (
            f"{CARS_URL}"
            f"?sort=newest"
            f"&page={page}"
            f"&page_size={PAGE_SIZE}"
        )

        print(
            f"\n[FETCH] "
            f"page={page}/{MAX_PAGES}"
        )

        result, api_key = request_api(
            url,
            api_key
        )

        data = result.get(
            "data",
            []
        )

        if not data:

            print(
                "[FETCH] 데이터 없음 -> 종료"
            )

            break

        fetched += len(data)

        # -----------------------------------------
        # 해당 페이지 바로 DB 적재
        # -----------------------------------------

        for raw in data:

            car_id = raw.get("id")

            # 같은 실행에서 중복 응답 확인
            if car_id in seen_car_ids:

                print(
                    f"[WARN] 중복 car_id 발견: "
                    f"{car_id}"
                )

            else:

                seen_car_ids.add(
                    car_id
                )

            try:

                # FK 때문에 업무영역 먼저
                upsert_business_area(
                    cursor,
                    raw
                )

                car = normalize_car(
                    raw
                )

                result_type = upsert_car(
                    cursor,
                    car
                )

                if result_type == "inserted":

                    inserted += 1

                else:

                    updated += 1

            except Exception as e:

                failed += 1

                print(
                    f"[ERROR] "
                    f"car_id={car_id} "
                    f"{e}"
                )

        # -----------------------------------------
        # 100건 단위 COMMIT
        # -----------------------------------------

        conn.commit()

        print(
            f"[LOAD] page={page}"
            f" | API누적={fetched}"
            f" | 고유차량={len(seen_car_ids)}"
            f" | 신규={inserted}"
            f" | 수정={updated}"
            f" | 실패={failed}"
        )

        # 서버 부하 방지
        time.sleep(2.0)

    return (
        fetched,
        inserted,
        updated,
        failed,
        len(seen_car_ids)
    )


# =========================================================
# 10. MAIN
# =========================================================

def main():

    started_at = datetime.now()

    conn = None
    cursor = None

    fetched = 0
    inserted = 0
    updated = 0
    failed = 0

    try:

        # -----------------------------------------
        # MySQL
        # -----------------------------------------

        conn = pymysql.connect(
            **MYSQL_CONFIG
        )

        print("[MYSQL] 연결 완료")

        create_tables(conn)

        cursor = conn.cursor()

        # -----------------------------------------
        # API KEY
        # -----------------------------------------

        api_key = get_api_key()

        print("\n==============================")
        print(" 최신 차량 10,000건 초기 적재")
        print("==============================\n")

        (
            fetched,
            inserted,
            updated,
            failed,
            unique_count
        ) = load_initial_cars(
            conn,
            cursor,
            api_key
        )

        finished_at = datetime.now()

        status = (
            "SUCCESS"
            if failed == 0
            else "PARTIAL_SUCCESS"
        )

        write_log(
            conn,
            started_at,
            finished_at,
            fetched,
            inserted,
            updated,
            failed,
            status
        )

        print("\n==============================")
        print(" 초기 적재 완료")
        print("==============================")

        print(
            f"API 조회     : {fetched}"
        )

        print(
            f"고유 car_id   : {unique_count}"
        )

        print(
            f"신규 INSERT  : {inserted}"
        )

        print(
            f"기존 UPDATE  : {updated}"
        )

        print(
            f"실패         : {failed}"
        )

        print(
            f"상태         : {status}"
        )

        # -----------------------------------------
        # 검증 경고
        # -----------------------------------------

        if unique_count != 10000:

            print(
                "\n[WARN] "
                "고유 차량 수가 정확히 10,000건이 아닙니다."
            )

            print(
                "페이지 조회 중 신규 데이터가 추가되면서 "
                "페이지 경계가 이동했을 가능성이 있습니다."
            )

    except Exception as e:

        finished_at = datetime.now()

        print(
            f"\n[FATAL ERROR] {e}"
        )

        if conn:

            try:

                write_log(
                    conn,
                    started_at,
                    finished_at,
                    fetched,
                    inserted,
                    updated,
                    failed + 1,
                    "FAILED",
                    str(e)
                )

            except Exception:
                pass

    finally:

        if cursor:
            cursor.close()

        if conn:
            conn.close()

        print("[MYSQL] 연결 종료")


# =========================================================
# 실행
# =========================================================

if __name__ == "__main__":
    main()

[MYSQL] 연결 완료

 전체 차량 API 수집 시작

[API KEY] 현재 API Key 조회 완료
[FETCH] page=1 url=http://192.168.0.51:4000/api/v1/cars/cursor?after_id=0&limit=100
[FETCH] 이번 조회: 100건 / 누적: 100건
[FETCH] page=2 url=http://192.168.0.51:4000/api/v1/cars/cursor?after_id=100&until_id=107388&limit=100&dataset_epoch=aa07d487-1652-406c-8474-f1f323bf5cb7
[FETCH] 이번 조회: 100건 / 누적: 200건
[FETCH] page=3 url=http://192.168.0.51:4000/api/v1/cars/cursor?after_id=200&until_id=107388&limit=100&dataset_epoch=aa07d487-1652-406c-8474-f1f323bf5cb7
[FETCH] 이번 조회: 100건 / 누적: 300건
[FETCH] page=4 url=http://192.168.0.51:4000/api/v1/cars/cursor?after_id=300&until_id=107388&limit=100&dataset_epoch=aa07d487-1652-406c-8474-f1f323bf5cb7
[FETCH] 이번 조회: 100건 / 누적: 400건
[FETCH] page=5 url=http://192.168.0.51:4000/api/v1/cars/cursor?after_id=400&until_id=107388&limit=100&dataset_epoch=aa07d487-1652-406c-8474-f1f323bf5cb7
[FETCH] 이번 조회: 100건 / 누적: 500건
[FETCH] page=6 url=http://192.168.0.51:4000/api/v1/cars/cursor?after_id=500&until_id=1073

KeyboardInterrupt: 